# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 10

In this version, we extend the single Transformer block from Version 9 into a small decoder-only Transformer language model.

The model keeps the same character vocabulary, fixed four-character context, trainable character embeddings, positional embeddings and causal autoregressive objective used in the previous version.

Version 9 introduced the internal structure of one Transformer block:

`character + positional embeddings`  
↓  
`causal self-attention`  
↓  
`residual connection + normalization`  
↓  
`feed-forward network`  
↓  
`residual connection + normalization`

Version 10 makes this structure reusable.

Instead of managing every Transformer parameter separately, the attention, residual connections, normalization and feed-forward operations are grouped into a reusable Transformer block.

Multiple Transformer blocks can then be stacked sequentially:

`character + positional embeddings`  
↓  
`Transformer block 1`  
↓  
`Transformer block 2`  
↓  
`vocabulary logits`

Each block receives the sequence representation produced by the previous block and refines it while preserving causal information flow.

The causal mask still prevents every position from attending to future positions, and every sequence position continues to learn the next-character prediction.

During generation, only the logits from the final context position are used to sample the next character.

This version introduces the basic structure of a small decoder-only Transformer while also simplifying the implementation through reusable model components.

## 1. Imports and Configuration

The Python standard library configures the execution environment before TensorFlow is imported.

GPU execution is disabled because this small model runs efficiently on the CPU and does not require CUDA. Low-level TensorFlow logs are suppressed to keep the notebook output clean.

TensorFlow provides tensor operations, trainable variables and automatic differentiation.

Python classes organize the Transformer operations and parameters into reusable model components while keeping the calculations explicit.

NumPy remains useful for reproducible data shuffling and sampling, while TensorFlow performs the model calculations and training.

The configuration collects the values that control the experiment.

`CONTEXT_LENGTH` defines the number of character positions processed together.

`EMBEDDING_DIM` defines the size of the trainable character and positional representations and therefore the main representation size of the model.

`ATTENTION_DIM` defines the size of the query, key and value vectors inside each Transformer block.

`FEED_FORWARD_DIM` defines the internal size of the position-wise feed-forward network.

`NUM_BLOCKS` defines how many Transformer blocks are stacked sequentially. Version 10 uses two blocks.

`LAYER_NORM_EPSILON` provides numerical stability during normalization.

An explicit seed makes weight initialization, data splitting, mini-batch shuffling and text generation reproducible.

The main architectural change is that repeated Transformer operations are now grouped into reusable Python components instead of managing every parameter separately.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import tensorflow as tf

tf.config.set_visible_devices([], "GPU")

In [2]:
SEED = 42
TRAIN_FRACTION = 0.8
BATCH_SIZE = 32
LEARNING_RATE = 1.0
EPOCHS = 100

CONTEXT_LENGTH = 4
EMBEDDING_DIM = 8
ATTENTION_DIM = 16
FEED_FORWARD_DIM = 16
NUM_BLOCKS = 2
LAYER_NORM_EPSILON = 1e-5

tf.keras.utils.set_random_seed(SEED)

## 2. Training Data

### Training text

The English corpus is included directly in the notebook. It provides the text from which the model learns character patterns.

The corpus is kept unchanged from Version 9 so that the effect of stacking multiple Transformer blocks can be observed without changing the training data.

Keeping the same corpus also makes the behavior of the new decoder-only architecture easier to compare with the previous versions.

In [3]:
corpus = 'language models learn patterns from text.\na small model predicts what character may come next.\nwe begin with counting because counting is easy to inspect.\nthe model sees letters, spaces, and punctuation.\neach prediction comes from examples found in the training text.\nsimple systems help us understand more advanced systems.\nlater versions will learn parameters with neural networks.\nclear experiments make machine learning easier to study.'

print(corpus)
print("Characters:", len(corpus))

language models learn patterns from text.
a small model predicts what character may come next.
we begin with counting because counting is easy to inspect.
the model sees letters, spaces, and punctuation.
each prediction comes from examples found in the training text.
simple systems help us understand more advanced systems.
later versions will learn parameters with neural networks.
clear experiments make machine learning easier to study.
Characters: 440


### Vocabulary

The vocabulary is the set of symbols the model can represent. Because this is a character-level model, every letter, space, punctuation mark and newline is a token.

Each character is assigned an integer identifier.

As in Version 9, these identifiers are used to look up trainable character embedding vectors.

The vocabulary remains unchanged so that Version 10 modifies the Transformer architecture without changing the representation of the training text.

In [4]:
vocabulary = sorted(set(corpus))
vocabulary_size = len(vocabulary)

character_to_id = {character: index for index, character in enumerate(vocabulary)}
id_to_character = {index: character for character, index in character_to_id.items()}

print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", repr("".join(vocabulary)))
print("First mappings:", list(character_to_id.items())[:10])

Vocabulary size: 27
Vocabulary: '\n ,.abcdefghiklmnoprstuvwxy'
First mappings: [('\n', 0), (' ', 1), (',', 2), ('.', 3), ('a', 4), ('b', 5), ('c', 6), ('d', 7), ('e', 8), ('f', 9)]


### Context windows and numerical encoding

Each character is converted into its integer identifier.

For the text `modeling`:

`modeling`  
↓  
`[15, 17, 7, 8, 14, 12, 16, 10]`

The model keeps the fixed four-character context used in Version 9.

Every input position predicts the character that immediately follows it.

For example:

`mode`  
↓  
`odel`

This creates four next-character predictions:

- `m -> o`
- `mo -> d`
- `mod -> e`
- `mode -> l`

Numerically:

`[15, 17, 7, 8]`  
↓  
`[17, 7, 8, 14]`

The input and target therefore both contain four character identifiers.

The same causal constraint introduced in Version 9 is preserved throughout every Transformer block.

Each position can use only itself and earlier positions when making its prediction, regardless of how many Transformer blocks are stacked.

The final position predicts the character that follows the complete four-character context.

During autoregressive generation, only the prediction produced by the final position is sampled before the context window moves forward.

Keeping the same input-target structure allows Version 10 to focus specifically on the transition from one Transformer block to a stack of reusable Transformer blocks.

In [5]:
examples = [
    (
        corpus[index:index + CONTEXT_LENGTH],
        corpus[index + 1:index + CONTEXT_LENGTH + 1]
    )
    for index in range(len(corpus) - CONTEXT_LENGTH)
]

input_ids = np.array([
    [character_to_id[character] for character in context]
    for context, _ in examples
], dtype=np.int32)

target_ids = np.array([
    [character_to_id[character] for character in targets]
    for _, targets in examples
], dtype=np.int32)

print("Number of examples:", len(examples))
print("Input shape:", input_ids.shape)
print("Target shape:", target_ids.shape)
print("First 8 examples:", examples[:8])
print("First 8 input IDs:")
print(input_ids[:8])
print("First 8 target IDs:")
print(target_ids[:8])

Number of examples: 436
Input shape: (436, 4)
Target shape: (436, 4)
First 8 examples: [('lang', 'angu'), ('angu', 'ngua'), ('ngua', 'guag'), ('guag', 'uage'), ('uage', 'age '), ('age ', 'ge m'), ('ge m', 'e mo'), ('e mo', ' mod')]
First 8 input IDs:
[[14  4 16 10]
 [ 4 16 10 22]
 [16 10 22  4]
 [10 22  4 10]
 [22  4 10  8]
 [ 4 10  8  1]
 [10  8  1 15]
 [ 8  1 15 17]]
First 8 target IDs:
[[ 4 16 10 22]
 [16 10 22  4]
 [10 22  4 10]
 [22  4 10  8]
 [ 4 10  8  1]
 [10  8  1 15]
 [ 8  1 15 17]
 [ 1 15 17  7]]


## 3. Neural Model

### Reusable Transformer blocks

Version 10 turns the single explicit Transformer block from Version 9 into a small decoder-only Transformer language model.

The internal mathematics of each block remains unchanged:

`input representation`  
↓  
`Q / K / V projections`  
↓  
`causal scaled dot-product attention`  
↓  
`attention output projection`  
↓  
`residual connection + layer normalization`  
↓  
`feed-forward network`  
↓  
`residual connection + layer normalization`

The main change is organizational.

Instead of keeping every Transformer parameter as a separate top-level variable and passing all of them manually between functions, the parameters and operations belonging to one block are grouped inside a reusable `TransformerBlock`.

Version 10 creates two independent Transformer blocks and applies them sequentially:

`character + positional embeddings`  
↓  
`Transformer block 1`  
↓  
`Transformer block 2`  
↓  
`vocabulary logits`

Each block has its own trainable query, key, value, attention output, feed-forward and normalization parameters.

The output of the first block becomes the input of the second block.

The causal mask is applied independently inside every block, so no position can receive information from future positions at any depth of the model.

A `DecoderOnlyLanguageModel` then groups:

- the character embeddings;
- the positional embeddings;
- the stack of Transformer blocks;
- the final vocabulary projection.

This reduces parameter passing and makes the architecture easier to extend while keeping the Transformer calculations explicit and inspectable.

In [6]:
inputs = tf.convert_to_tensor(input_ids, dtype=tf.int32)
targets = tf.convert_to_tensor(target_ids, dtype=tf.int32)


def layer_normalize(values, scale, shift):
    mean = tf.reduce_mean(values, axis=-1, keepdims=True)
    variance = tf.reduce_mean(tf.square(values - mean), axis=-1, keepdims=True)
    normalized = (values - mean) / tf.sqrt(variance + LAYER_NORM_EPSILON)
    return normalized * scale + shift


class TransformerBlock:
    def __init__(self, model_random, block_index):
        prefix = f"block_{block_index}"

        self.query_weights = tf.Variable(
            model_random.normal((EMBEDDING_DIM, ATTENTION_DIM), mean=0.0, stddev=0.01, dtype=tf.float32),
            name=f"{prefix}_query_weights"
        )
        self.key_weights = tf.Variable(
            model_random.normal((EMBEDDING_DIM, ATTENTION_DIM), mean=0.0, stddev=0.01, dtype=tf.float32),
            name=f"{prefix}_key_weights"
        )
        self.value_weights = tf.Variable(
            model_random.normal((EMBEDDING_DIM, ATTENTION_DIM), mean=0.0, stddev=0.01, dtype=tf.float32),
            name=f"{prefix}_value_weights"
        )
        self.attention_output_weights = tf.Variable(
            model_random.normal((ATTENTION_DIM, EMBEDDING_DIM), mean=0.0, stddev=0.01, dtype=tf.float32),
            name=f"{prefix}_attention_output_weights"
        )

        self.layer_norm_1_scale = tf.Variable(tf.ones((EMBEDDING_DIM,), dtype=tf.float32), name=f"{prefix}_layer_norm_1_scale")
        self.layer_norm_1_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,), dtype=tf.float32), name=f"{prefix}_layer_norm_1_shift")

        self.feed_forward_input_weights = tf.Variable(
            model_random.normal((EMBEDDING_DIM, FEED_FORWARD_DIM), mean=0.0, stddev=0.01, dtype=tf.float32),
            name=f"{prefix}_feed_forward_input_weights"
        )
        self.feed_forward_output_weights = tf.Variable(
            model_random.normal((FEED_FORWARD_DIM, EMBEDDING_DIM), mean=0.0, stddev=0.01, dtype=tf.float32),
            name=f"{prefix}_feed_forward_output_weights"
        )

        self.layer_norm_2_scale = tf.Variable(tf.ones((EMBEDDING_DIM,), dtype=tf.float32), name=f"{prefix}_layer_norm_2_scale")
        self.layer_norm_2_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,), dtype=tf.float32), name=f"{prefix}_layer_norm_2_shift")

    @property
    def trainable_variables(self):
        return [
            self.query_weights, self.key_weights, self.value_weights,
            self.attention_output_weights,
            self.layer_norm_1_scale, self.layer_norm_1_shift,
            self.feed_forward_input_weights, self.feed_forward_output_weights,
            self.layer_norm_2_scale, self.layer_norm_2_shift
        ]

    def __call__(self, values):
        queries = tf.matmul(values, self.query_weights)
        keys = tf.matmul(values, self.key_weights)
        projected_values = tf.matmul(values, self.value_weights)

        attention_scores = tf.matmul(queries, keys, transpose_b=True)
        scale = tf.sqrt(tf.cast(ATTENTION_DIM, tf.float32))
        scaled_attention_scores = attention_scores / scale

        causal_mask = tf.linalg.band_part(tf.ones((CONTEXT_LENGTH, CONTEXT_LENGTH), dtype=tf.float32), -1, 0)
        masked_attention_scores = tf.where(
            causal_mask == 1.0,
            scaled_attention_scores,
            tf.constant(-1e9, dtype=tf.float32)
        )

        attention_weights = tf.nn.softmax(masked_attention_scores, axis=-1)
        attention_output = tf.matmul(attention_weights, projected_values)
        projected_attention = tf.matmul(attention_output, self.attention_output_weights)

        attention_residual = values + projected_attention
        normalized_attention = layer_normalize(
            attention_residual, self.layer_norm_1_scale, self.layer_norm_1_shift
        )

        feed_forward_hidden = tf.nn.relu(tf.matmul(normalized_attention, self.feed_forward_input_weights))
        feed_forward_output = tf.matmul(feed_forward_hidden, self.feed_forward_output_weights)

        feed_forward_residual = normalized_attention + feed_forward_output
        block_output = layer_normalize(
            feed_forward_residual, self.layer_norm_2_scale, self.layer_norm_2_shift
        )

        return block_output, attention_weights


class DecoderOnlyLanguageModel:
    def __init__(self, vocabulary_size, seed):
        model_random = tf.random.Generator.from_seed(seed)

        self.embedding_matrix = tf.Variable(
            model_random.normal((vocabulary_size, EMBEDDING_DIM), mean=0.0, stddev=0.01, dtype=tf.float32),
            name="embedding_matrix"
        )
        self.position_embedding_matrix = tf.Variable(
            model_random.normal((CONTEXT_LENGTH, EMBEDDING_DIM), mean=0.0, stddev=0.01, dtype=tf.float32),
            name="position_embedding_matrix"
        )

        self.transformer_blocks = [
            TransformerBlock(model_random, block_index)
            for block_index in range(NUM_BLOCKS)
        ]

        self.output_weights = tf.Variable(
            model_random.normal((EMBEDDING_DIM, vocabulary_size), mean=0.0, stddev=0.01, dtype=tf.float32),
            name="output_weights"
        )

    @property
    def trainable_variables(self):
        variables = [self.embedding_matrix, self.position_embedding_matrix]

        for block in self.transformer_blocks:
            variables.extend(block.trainable_variables)

        variables.append(self.output_weights)
        return variables

    def __call__(self, inputs):
        embeddings = tf.gather(self.embedding_matrix, inputs)
        values = embeddings + self.position_embedding_matrix[tf.newaxis, :, :]

        attention_weights = []

        for block in self.transformer_blocks:
            values, block_attention = block(values)
            attention_weights.append(block_attention)

        logits = tf.matmul(values, self.output_weights)
        return values, attention_weights, logits


model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)

### Model structure and trainable parameters

The decoder-only model now contains shared input and output parameters together with two independent Transformer blocks.

The shared parameters are:

- character embeddings;
- positional embeddings;
- final vocabulary projection.

Each Transformer block contains its own:

- query projection;
- key projection;
- value projection;
- attention output projection;
- two LayerNorm scale and shift pairs;
- feed-forward input projection;
- feed-forward output projection.

Because the two blocks have the same architecture but independent parameters, stacking them increases model depth without changing the representation size.

Version 10 therefore contains:

`shared parameters`  
`+ Transformer block 1`  
`+ Transformer block 2`

The model has 2064 trainable parameters in total.

Inspecting the tensor shapes also verifies that both Transformer blocks preserve the `(batch, 4, 8)` sequence representation required by the next block.

In [7]:
model_output, attention_weights, logits = model(inputs[:1])

print("Number of Transformer blocks:", len(model.transformer_blocks))
print("Number of trainable variables:", len(model.trainable_variables))

print("\nInput shape:", inputs[:1].shape)
print("Model output shape:", model_output.shape)
print("Logits shape:", logits.shape)

for block_index, block_attention in enumerate(attention_weights, start=1):
    print(f"Block {block_index} attention shape:", block_attention.shape)

print("\nTrainable variables:")

total_parameters = 0

for variable in model.trainable_variables:
    parameter_count = int(tf.size(variable))
    total_parameters += parameter_count
    print(f"{variable.name:40s} {str(tuple(variable.shape)):15s} {parameter_count}")

print("\nTotal trainable parameters:", total_parameters)

Number of Transformer blocks: 2
Number of trainable variables: 23

Input shape: (1, 4)
Model output shape: (1, 4, 8)
Logits shape: (1, 4, 27)
Block 1 attention shape: (1, 4, 4)
Block 2 attention shape: (1, 4, 4)

Trainable variables:
embedding_matrix:0                       (27, 8)         216
position_embedding_matrix:0              (4, 8)          32
block_0_query_weights:0                  (8, 16)         128
block_0_key_weights:0                    (8, 16)         128
block_0_value_weights:0                  (8, 16)         128
block_0_attention_output_weights:0       (16, 8)         128
block_0_layer_norm_1_scale:0             (8,)            8
block_0_layer_norm_1_shift:0             (8,)            8
block_0_feed_forward_input_weights:0     (8, 16)         128
block_0_feed_forward_output_weights:0    (16, 8)         128
block_0_layer_norm_2_scale:0             (8,)            8
block_0_layer_norm_2_shift:0             (8,)            8
block_1_query_weights:0                  (8

### Training and validation split

The examples are divided into two separate groups:

- the training set is used to update the model parameters;
- the validation set is used to measure the loss on examples that do not update the parameters.

Each input example contains four character identifiers and each target contains the four following character identifiers.

The same reproducible train/validation split used in Version 9 is preserved.

The split is intentionally kept unchanged so that Version 10 focuses on the architectural change from one Transformer block to two stacked reusable blocks.

Inside both Transformer blocks, causal attention ensures that every prediction can use only the current and previous input positions.

The indices are shuffled with a local random generator, making the split reproducible.

In [8]:
split_random = np.random.default_rng(SEED)

indices = split_random.permutation(len(inputs))
split_position = int(len(indices) * TRAIN_FRACTION)

train_indices = indices[:split_position]
validation_indices = indices[split_position:]

train_inputs = tf.gather(inputs, train_indices)
train_targets = tf.gather(targets, train_indices)

validation_inputs = tf.gather(inputs, validation_indices)
validation_targets = tf.gather(targets, validation_indices)

print("Training examples:", len(train_inputs))
print("Validation examples:", len(validation_inputs))
print("Training input shape:", train_inputs.shape)
print("Training target shape:", train_targets.shape)
print("Validation input shape:", validation_inputs.shape)
print("Validation target shape:", validation_targets.shape)

Training examples: 348
Validation examples: 88
Training input shape: (348, 4)
Training target shape: (348, 4)
Validation input shape: (88, 4)
Validation target shape: (88, 4)


### Softmax probabilities

TensorFlow provides `tf.nn.softmax` to convert logits into probabilities.

After the input passes through both Transformer blocks, the model produces one vocabulary distribution for every context position.

For each position:

- every probability is between 0 and 1;
- the probabilities over the vocabulary sum to 1;
- higher logits produce higher probabilities.

Softmax is applied along the final vocabulary dimension.

TensorFlow handles the numerical stability of this operation internally.

In [9]:
def softmax(logits):
    return tf.nn.softmax(logits, axis=-1)

### Cross-entropy loss

Cross-entropy measures how much probability the model assigns to the correct next character.

The decoder-only Transformer produces next-character logits for all four context positions.

For an input such as:

`mode`

the corresponding targets are:

`odel`

The loss therefore measures four predictions.

TensorFlow calculates sparse softmax cross-entropy directly from the logits and integer target IDs using a numerically stable operation.

The individual losses contain one value for every example and every context position.

Their mean produces one scalar loss used for training.

Because the model is now organized into reusable components, the loss function only needs the model, inputs and targets.

Gradients automatically flow through the vocabulary projection, both Transformer blocks, positional embeddings and character embeddings.

In [10]:
def calculate_loss(model, inputs, targets):
    _, _, logits = model(inputs)
    example_losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)
    return tf.reduce_mean(example_losses)

In [11]:
initial_train_loss = calculate_loss(model, train_inputs, train_targets)
initial_validation_loss = calculate_loss(model, validation_inputs, validation_targets)

print("Initial train loss:", initial_train_loss.numpy())
print("Initial validation loss:", initial_validation_loss.numpy())

Initial train loss: 3.2953002
Initial validation loss: 3.291214


### Training with mini-batch gradient descent

Training remains organized into epochs and mini-batches.

At each recorded epoch, training and validation loss are measured using the current model parameters.

Except after the final recorded epoch, the training examples are shuffled and divided into mini-batches.

For every mini-batch:

1. `tf.GradientTape` records the complete decoder-only Transformer computation;
2. TensorFlow calculates gradients automatically through both Transformer blocks;
3. every trainable model variable is updated manually with gradient descent.

The validation examples are never used to update the model.

Gradients flow backward through the vocabulary projection, both Transformer blocks, positional embeddings and character embeddings.

Embedding lookups can produce sparse `IndexedSlices` gradients. Because this model is very small, gradients are converted to regular dense tensors before the manual parameter update.

Because the parameters are now grouped inside reusable model components, the training procedure no longer needs to manage every Transformer variable separately.

### Mini-batches

A mini-batch is a small group of training examples.

The model updates all of its trainable variables after every mini-batch instead of processing the complete training set at once.

The training examples are shuffled reproducibly before every epoch.

The final mini-batch may contain fewer examples than the configured batch size.

In [12]:
def create_batches(inputs, targets, batch_size):
    for start in range(0, len(inputs), batch_size):
        end = start + batch_size

        batch_inputs = inputs[start:end]
        batch_targets = targets[start:end]

        yield batch_inputs, batch_targets

In [13]:
def train_model(train_inputs, train_targets, validation_inputs, validation_targets,
                vocabulary_size, learning_rate=1.0, batch_size=32, epochs=100,
                seed=42, print_every=10):

    trained_model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=seed)
    training_random = np.random.default_rng(seed)
    trainable_variables = trained_model.trainable_variables

    train_loss_history = []
    validation_loss_history = []

    for epoch in range(epochs + 1):
        train_loss = float(calculate_loss(trained_model, train_inputs, train_targets))
        validation_loss = float(calculate_loss(trained_model, validation_inputs, validation_targets))

        train_loss_history.append(train_loss)
        validation_loss_history.append(validation_loss)

        if print_every is not None and (epoch % print_every == 0 or epoch == epochs):
            print(f"Epoch {epoch:3d} | train loss {train_loss:.6f} | validation loss {validation_loss:.6f}")

        if epoch == epochs:
            break

        shuffled_indices = training_random.permutation(len(train_inputs))
        shuffled_inputs = tf.gather(train_inputs, shuffled_indices)
        shuffled_targets = tf.gather(train_targets, shuffled_indices)

        for batch_inputs, batch_targets in create_batches(shuffled_inputs, shuffled_targets, batch_size):
            with tf.GradientTape() as tape:
                batch_loss = calculate_loss(trained_model, batch_inputs, batch_targets)

            gradients = tape.gradient(batch_loss, trainable_variables)

            for variable, gradient in zip(trainable_variables, gradients):
                variable.assign_sub(learning_rate * tf.convert_to_tensor(gradient))

    return trained_model, train_loss_history, validation_loss_history

In [14]:
trained_model, train_loss_history, validation_loss_history = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    vocabulary_size=vocabulary_size,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED
)

final_train_loss = train_loss_history[-1]
final_validation_loss = validation_loss_history[-1]

best_validation_epoch = int(np.argmin(validation_loss_history))
best_validation_loss = validation_loss_history[best_validation_epoch]

print()
print("Initial train loss:", initial_train_loss.numpy())
print("Final train loss:", final_train_loss)
print("Initial validation loss:", initial_validation_loss.numpy())
print("Final validation loss:", final_validation_loss)
print("Best validation loss:", best_validation_loss)
print("Best validation epoch:", best_validation_epoch)

Epoch   0 | train loss 3.295300 | validation loss 3.291214
Epoch  10 | train loss 2.141560 | validation loss 2.297166
Epoch  20 | train loss 2.067055 | validation loss 2.249319
Epoch  30 | train loss 1.996058 | validation loss 2.201702
Epoch  40 | train loss 2.066956 | validation loss 2.270509
Epoch  50 | train loss 2.003075 | validation loss 2.265693
Epoch  60 | train loss 1.919792 | validation loss 2.212168
Epoch  70 | train loss 1.870389 | validation loss 2.170606
Epoch  80 | train loss 1.832215 | validation loss 2.157501
Epoch  90 | train loss 1.889135 | validation loss 2.197083
Epoch 100 | train loss 1.892438 | validation loss 2.220907

Initial train loss: 3.2953002
Final train loss: 1.8924381732940674
Initial validation loss: 3.291214
Final validation loss: 2.220906972885132
Best validation loss: 2.141364336013794
Best validation epoch: 68


### Reading the losses

At initialization, training and validation loss are close to the value expected from an almost uniform probability distribution over the 27-character vocabulary.

During training, the decoder-only Transformer learns next-character relationships by updating the character embeddings, positional embeddings, both Transformer blocks and the final vocabulary projection.

The training cell above reports:

- initial and final training loss;
- initial and final validation loss;
- the best validation loss;
- the epoch at which the best validation loss occurs.

If training loss continues to decrease while validation loss starts to increase, the model is beginning to overfit the small training corpus.

The exact behavior is determined by the values printed by the training cell.

Version 10 keeps the same sequential next-character objective introduced in Version 9, so all four context positions contribute to the loss.

The main architectural difference is therefore the depth of the model: the sequence now passes through two stacked Transformer blocks instead of one.

### Learned probabilities

After training, the decoder-only Transformer produces a next-character probability distribution for every position in the context.

Character and positional embeddings create the initial sequence representation.

The representation then passes through both Transformer blocks in order.

Each block applies causal self-attention, residual connections, layer normalization and a feed-forward network before passing its output to the next block.

During training, all four sequence positions contribute to the loss.

During generation, only the probability distribution produced by the final context position is used to sample the next character.

The example below inspects the final-position probability distribution for the context `mode`.

In [15]:
example_context = "mode"
example_context_ids = tf.constant([[character_to_id[character] for character in example_context]], dtype=tf.int32)

example_model_output, example_attention_weights, example_logits = trained_model(example_context_ids)

learned_probabilities = softmax(example_logits)[0, -1].numpy()

sorted_probabilities = sorted(
    zip(vocabulary, learned_probabilities),
    key=lambda item: item[1],
    reverse=True
)

print("Context:", repr(example_context))
print("Model output shape:", example_model_output.shape)
print("Number of attention matrices:", len(example_attention_weights))
print("Logits shape:", example_logits.shape)
print()

for character, probability in sorted_probabilities:
    print(repr(character), round(float(probability), 4))

print()
print("Total probability:", learned_probabilities.sum())

Context: 'mode'
Model output shape: (1, 4, 8)
Number of attention matrices: 2
Logits shape: (1, 4, 27)

' ' 0.4307
't' 0.1878
'l' 0.106
'r' 0.0732
'c' 0.0668
's' 0.0394
'd' 0.0273
'e' 0.0125
'g' 0.0118
'u' 0.0085
'n' 0.0077
'a' 0.0076
'p' 0.0065
'm' 0.004
'x' 0.0032
'y' 0.0019
'i' 0.0019
'k' 0.0018
'.' 0.0011
',' 0.0002
'b' 0.0002
'w' 0.0
'\n' 0.0
'f' 0.0
'v' 0.0
'o' 0.0
'h' 0.0

Total probability: 1.0


### Inspecting attention across stacked Transformer blocks

Version 10 contains two independent causal self-attention operations, one inside each Transformer block.

For the same input context, each block produces its own `4 × 4` attention matrix.

The first block operates directly on the character and positional representations.

The second block receives the transformed sequence produced by the first block and can therefore learn a different attention pattern.

The causal rule remains identical in both blocks:

- position 0 can attend only to position 0;
- position 1 can attend to positions 0 and 1;
- position 2 can attend to positions 0, 1 and 2;
- position 3 can attend to all four positions.

Every value above the main diagonal must remain zero.

Inspecting both matrices shows how stacked Transformer blocks can process the same sequence differently while preserving causal information flow.

In [16]:
print("Context:", repr(example_context))
print()

for block_index, block_attention in enumerate(example_attention_weights, start=1):
    attention_matrix = block_attention[0].numpy()

    print(f"Transformer block {block_index}")
    print("   ", " ".join(f"{character:>7s}" for character in example_context))

    for row_index, character in enumerate(example_context):
        row = " ".join(f"{value:7.4f}" for value in attention_matrix[row_index])
        print(f"{character:>2s} {row}")

    print()

Context: 'mode'

Transformer block 1
          m       o       d       e
 m  1.0000  0.0000  0.0000  0.0000
 o  0.4886  0.5114  0.0000  0.0000
 d  0.3881  0.2613  0.3506  0.0000
 e  0.3556  0.1382  0.2994  0.2068

Transformer block 2
          m       o       d       e
 m  1.0000  0.0000  0.0000  0.0000
 o  1.0000  0.0000  0.0000  0.0000
 d  0.1847  0.0006  0.8147  0.0000
 e  0.9981  0.0000  0.0019  0.0000



### Verifying causal attention

The causal mask must remain valid inside every Transformer block.

For each attention matrix:

- all values above the main diagonal must be zero;
- every row must sum to approximately 1;
- all attention values must remain finite.

Because both blocks apply the same causal restriction independently, stacking them does not allow future information to leak into earlier positions.

In [17]:
for block_index, block_attention in enumerate(example_attention_weights, start=1):
    attention_matrix = block_attention[0].numpy()
    future_values = attention_matrix[np.triu_indices(CONTEXT_LENGTH, k=1)]
    row_sums = attention_matrix.sum(axis=1)

    print(f"Block {block_index}")
    print("Maximum future attention:", float(np.max(np.abs(future_values))))
    print("Row sums:", row_sums)
    print("All values finite:", np.isfinite(attention_matrix).all())
    print()

Block 1
Maximum future attention: 0.0
Row sums: [1.        1.        1.        1.0000001]
All values finite: True

Block 2
Maximum future attention: 0.0
Row sums: [1. 1. 1. 1.]
All values finite: True



## 4. Autoregressive Generation

The trained decoder-only Transformer generates text one character at a time.

Generation starts from a four-character context.

At every step:

1. the current context is converted into character IDs;
2. the complete context passes through both Transformer blocks;
3. the model produces vocabulary logits at all four positions;
4. only the logits from the final position are converted into probabilities;
5. one character is sampled from that distribution;
6. the sampled character is appended to the generated text;
7. the oldest character is removed from the context window.

The updated four-character window is then processed again.

Both Transformer blocks are therefore recomputed at every generation step.

Because causal attention is used throughout the model, the prediction at the final position depends only on characters available in the current context.

A local NumPy random generator makes sampling reproducible without modifying NumPy's global random state.

In [18]:
def generate_text(model, start_context, length, seed=42):
    if len(start_context) != CONTEXT_LENGTH:
        raise ValueError(f"start_context must contain exactly {CONTEXT_LENGTH} characters")

    generation_random = np.random.default_rng(seed)
    generated_text = start_context
    current_context = start_context

    while len(generated_text) < length:
        context_ids = tf.constant([[character_to_id[character] for character in current_context]], dtype=tf.int32)
        _, _, logits = model(context_ids)
        probabilities = softmax(logits)[0, -1].numpy()

        next_id = generation_random.choice(vocabulary_size, p=probabilities)
        next_character = id_to_character[next_id]

        generated_text += next_character
        current_context = generated_text[-CONTEXT_LENGTH:]

    return generated_text

### Sampling the next character

Before generating a longer sequence, the model can be inspected by sampling several independent next characters from the same context.

Every sample starts from the context `mode`.

The model calculates the same probability distribution each time, while the local random generator produces reproducible samples from that distribution.

In [19]:
sample_random = np.random.default_rng(SEED)

_, _, sample_logits = trained_model(example_context_ids)
sample_probabilities = softmax(sample_logits)[0, -1].numpy()

samples = [
    id_to_character[sample_random.choice(vocabulary_size, p=sample_probabilities)]
    for _ in range(5)
]

print("Context:", repr(example_context))
print("Five sampled next characters:", samples)

Context: 'mode'
Five sampled next characters: ['s', 'a', 't', 'r', ' ']


### Generating a longer sequence

The same sampling process can be repeated autoregressively to produce a longer character sequence.

The generated text begins with the supplied context and every new character becomes part of the next four-character input window.

The model still has a very small corpus and context length, so the output is expected to capture local character patterns rather than long-range linguistic coherence.

In [20]:
generated_text = generate_text(trained_model, start_context="mode", length=300, seed=SEED)

print(generated_text)

modes spaurs matrs in tromsin macortect.
lerns more nd  sprome comers ma mante ksprsins adval comain.
ausintrng comain ttrninim bomactcting varamatrim ne tude ndits tictctraung va.
worsind prandin tagrs intr  imn nsmattt.
we moult nimantsts s conctiong ins inin comnd tecn.
al ns inst fndeacom t.
ctt


## 5. Tests

These assertions verify the sequential context-target windows, reusable Transformer blocks, parameter shapes, causal attention, model outputs, probability distributions, gradients, training behavior and reproducibility.

The tests verify that:

- every input and target contains four character identifiers;
- the model contains exactly two Transformer blocks;
- the two Transformer blocks contain independent trainable variables;
- all 23 trainable variables are TensorFlow variables;
- the model contains exactly 2064 trainable parameters;
- both Transformer blocks preserve the expected sequence shape;
- every block produces a causal `4 × 4` attention matrix;
- future attention values remain zero in every block;
- every attention row sums to 1;
- training and validation sets remain separate;
- training reduces the loss and all recorded losses remain finite;
- next-character probabilities form a valid distribution;
- TensorFlow produces finite gradients for every trainable variable;
- every trainable variable changes during training;
- retraining with the same seed is reproducible;
- autoregressive generation is reproducible.

In [21]:
assert len(examples) == len(corpus) - CONTEXT_LENGTH
assert len(input_ids) == len(examples)
assert len(target_ids) == len(examples)

assert input_ids.shape == (len(examples), CONTEXT_LENGTH)
assert target_ids.shape == (len(examples), CONTEXT_LENGTH)

assert examples[0][0] == corpus[:CONTEXT_LENGTH]
assert examples[0][1] == corpus[1:CONTEXT_LENGTH + 1]
assert examples[1][0] == corpus[1:1 + CONTEXT_LENGTH]
assert examples[1][1] == corpus[2:2 + CONTEXT_LENGTH]

assert tf.is_tensor(inputs)
assert tf.is_tensor(targets)

# Model and parameter checks

initial_parameter_variables = model.trainable_variables
trained_parameter_variables = trained_model.trainable_variables

assert NUM_BLOCKS == 2
assert len(model.transformer_blocks) == NUM_BLOCKS
assert len(trained_model.transformer_blocks) == NUM_BLOCKS
assert len(initial_parameter_variables) == 23
assert len(trained_parameter_variables) == 23

assert all(isinstance(variable, tf.Variable) for variable in initial_parameter_variables)
assert all(isinstance(variable, tf.Variable) for variable in trained_parameter_variables)

assert model.embedding_matrix.shape == (vocabulary_size, EMBEDDING_DIM)
assert model.position_embedding_matrix.shape == (CONTEXT_LENGTH, EMBEDDING_DIM)
assert model.output_weights.shape == (EMBEDDING_DIM, vocabulary_size)

for block in model.transformer_blocks:
    assert block.query_weights.shape == (EMBEDDING_DIM, ATTENTION_DIM)
    assert block.key_weights.shape == (EMBEDDING_DIM, ATTENTION_DIM)
    assert block.value_weights.shape == (EMBEDDING_DIM, ATTENTION_DIM)
    assert block.attention_output_weights.shape == (ATTENTION_DIM, EMBEDDING_DIM)
    assert block.layer_norm_1_scale.shape == (EMBEDDING_DIM,)
    assert block.layer_norm_1_shift.shape == (EMBEDDING_DIM,)
    assert block.feed_forward_input_weights.shape == (EMBEDDING_DIM, FEED_FORWARD_DIM)
    assert block.feed_forward_output_weights.shape == (FEED_FORWARD_DIM, EMBEDDING_DIM)
    assert block.layer_norm_2_scale.shape == (EMBEDDING_DIM,)
    assert block.layer_norm_2_shift.shape == (EMBEDDING_DIM,)

# Transformer block independence

block_1_variables = model.transformer_blocks[0].trainable_variables
block_2_variables = model.transformer_blocks[1].trainable_variables

assert len(block_1_variables) == 10
assert len(block_2_variables) == 10
assert all(
    first_variable is not second_variable
    for first_variable, second_variable in zip(block_1_variables, block_2_variables)
)

for initial_variable, trained_variable in zip(initial_parameter_variables, trained_parameter_variables):
    assert trained_variable.shape == initial_variable.shape

# Trainable parameter count

parameters_per_block = (
    3 * EMBEDDING_DIM * ATTENTION_DIM
    + ATTENTION_DIM * EMBEDDING_DIM
    + 2 * EMBEDDING_DIM
    + EMBEDDING_DIM * FEED_FORWARD_DIM
    + FEED_FORWARD_DIM * EMBEDDING_DIM
    + 2 * EMBEDDING_DIM
)

expected_trainable_parameters = (
    vocabulary_size * EMBEDDING_DIM
    + CONTEXT_LENGTH * EMBEDDING_DIM
    + NUM_BLOCKS * parameters_per_block
    + EMBEDDING_DIM * vocabulary_size
)

actual_trainable_parameters = sum(int(tf.size(variable)) for variable in model.trainable_variables)

assert parameters_per_block == 800
assert expected_trainable_parameters == 2064
assert actual_trainable_parameters == expected_trainable_parameters

# Forward-pass and causal attention checks

test_context = tf.constant([[character_to_id[character] for character in "mode"]], dtype=tf.int32)
test_output, test_attention_weights, test_logits = trained_model(test_context)

assert test_output.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)
assert test_logits.shape == (1, CONTEXT_LENGTH, vocabulary_size)
assert len(test_attention_weights) == NUM_BLOCKS

future_positions = np.triu(np.ones((CONTEXT_LENGTH, CONTEXT_LENGTH), dtype=bool), k=1)

for block_attention in test_attention_weights:
    assert block_attention.shape == (1, CONTEXT_LENGTH, CONTEXT_LENGTH)
    assert np.all(np.isfinite(block_attention.numpy()))
    assert np.all(block_attention.numpy() >= 0.0)
    assert np.all(block_attention.numpy() <= 1.0)

    attention_row_sums = tf.reduce_sum(block_attention, axis=-1).numpy()
    assert np.allclose(attention_row_sums, np.ones((1, CONTEXT_LENGTH)), atol=1e-6)
    assert np.allclose(block_attention[0].numpy()[future_positions], 0.0, atol=1e-7)

assert np.all(np.isfinite(test_output.numpy()))
assert np.all(np.isfinite(test_logits.numpy()))

# Training and validation checks

assert len(train_inputs) + len(validation_inputs) == len(inputs)
assert train_inputs.shape == (len(train_inputs), CONTEXT_LENGTH)
assert train_targets.shape == (len(train_inputs), CONTEXT_LENGTH)
assert validation_inputs.shape == (len(validation_inputs), CONTEXT_LENGTH)
assert validation_targets.shape == (len(validation_inputs), CONTEXT_LENGTH)
assert len(np.intersect1d(train_indices, validation_indices)) == 0

assert len(train_loss_history) == EPOCHS + 1
assert len(validation_loss_history) == EPOCHS + 1
assert final_train_loss < float(initial_train_loss.numpy())
assert np.all(np.isfinite(train_loss_history))
assert np.all(np.isfinite(validation_loss_history))
assert best_validation_epoch == int(np.argmin(validation_loss_history))
assert best_validation_loss == validation_loss_history[best_validation_epoch]

# Probability checks

_, _, probability_logits = trained_model(test_context)
probability_values = softmax(probability_logits)[0, -1].numpy()

assert abs(probability_values.sum() - 1.0) < 1e-6
assert np.all(np.isfinite(probability_values))
assert np.all(probability_values >= 0.0)
assert np.all(probability_values <= 1.0)

# Gradient checks

gradient_inputs = train_inputs[:BATCH_SIZE]
gradient_targets = train_targets[:BATCH_SIZE]

with tf.GradientTape() as tape:
    gradient_loss = calculate_loss(model, gradient_inputs, gradient_targets)

gradients = tape.gradient(gradient_loss, model.trainable_variables)

assert all(gradient is not None for gradient in gradients)

dense_gradients = [tf.convert_to_tensor(gradient) for gradient in gradients]

for gradient, variable in zip(dense_gradients, model.trainable_variables):
    assert gradient.shape == variable.shape
    assert np.all(np.isfinite(gradient.numpy()))

# Parameter-update checks

for initial_variable, trained_variable in zip(initial_parameter_variables, trained_parameter_variables):
    assert np.max(np.abs(initial_variable.numpy() - trained_variable.numpy())) > 0.0

# Deterministic retraining

repeated_model, repeated_train_history, repeated_validation_history = train_model(
    train_inputs, train_targets, validation_inputs, validation_targets,
    vocabulary_size=vocabulary_size, learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE, epochs=EPOCHS, seed=SEED, print_every=None
)

for trained_variable, repeated_variable in zip(trained_model.trainable_variables, repeated_model.trainable_variables):
    assert np.allclose(trained_variable.numpy(), repeated_variable.numpy())

assert np.allclose(train_loss_history, repeated_train_history)
assert np.allclose(validation_loss_history, repeated_validation_history)

# Generation reproducibility

first_generation = generate_text(trained_model, start_context="mode", length=30, seed=10)
second_generation = generate_text(trained_model, start_context="mode", length=30, seed=10)

assert first_generation == second_generation
assert len(first_generation) == 30
assert first_generation.startswith("mode")

print("All checks passed.")

All checks passed.


## Notes

- The model remains a character-level neural language model.
- The training corpus and 27-character vocabulary remain unchanged from Version 9.
- Each example uses a fixed context of four characters.
- Every context position predicts the character that immediately follows it.
- Character identifiers are mapped to trainable 8-dimensional embeddings.
- Trainable positional embeddings preserve the order of the four context positions.
- Version 10 introduces a reusable `TransformerBlock` component.
- The decoder-only model stacks two independent Transformer blocks sequentially.
- Each Transformer block has its own query, key, value, attention output, feed-forward and LayerNorm parameters.
- Queries, keys and values use 16-dimensional attention representations.
- A causal mask is applied independently inside every Transformer block.
- The causal mask prevents every position from attending to future positions.
- Each attention output is projected back to the 8-dimensional model representation.
- Residual connections preserve information around both the attention and feed-forward operations.
- Layer normalization is applied after each residual connection.
- Each feed-forward network expands the representation from 8 to 16 values and projects it back to 8 values.
- The output of the first Transformer block becomes the input of the second Transformer block.
- Both blocks preserve the `(batch, 4, 8)` sequence representation.
- The complete model contains 23 trainable variables and 2064 trainable parameters.
- Every context position produces vocabulary logits during training.
- Cross-entropy loss is calculated over all four next-character predictions.
- During generation, only the logits from the final context position are used to sample the next character.
- Text generation remains autoregressive: each sampled character becomes part of the next four-character context.
- The implementation now groups model parameters inside reusable components instead of passing every Transformer parameter manually.
- Mini-batch creation is also reused explicitly during training instead of duplicating batching logic.
- Embedding lookups can produce sparse gradients, which are converted to dense tensors before the manual gradient-descent update because the model is very small.
- The model still uses one attention mechanism per Transformer block rather than multi-head attention.
- No bias terms are used in the attention, feed-forward or vocabulary projections.
- No dropout or learning-rate schedule is used.
- Neighboring context windows overlap, so the random train/validation split remains a didactic monitoring tool rather than a robust estimate of generalization to unseen text.
- Generation uses the parameters from the final training epoch rather than automatically restoring the parameters from the best validation epoch.
- Reproducibility is expected within the same execution environment; small numerical differences may appear across TensorFlow environments.
- The corpus remains extremely small and the context is limited to four characters, so generated text mainly reflects local character patterns.
- Version 10 completes the first small character-level decoder-only Transformer stage of the project.

The model is still intentionally far smaller than an LLM: it uses character tokens, a four-character context and a 440-character corpus.

Future versions will introduce new components and gradually evolve the architecture.